# comfio Smart Contract & ML Integration Tutorial

**Part 1** requires no optional extras — just the core package.
**Part 2** requires `pip install comfio[ml]` (scikit-learn), `comfio[torch]` (PyTorch), or `comfio[keras]` (TensorFlow).

This notebook covers:

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Compliance reporting & JSON export | — |
| 2 | Smart contract schema & ABI generation | — |
| 3 | Contract payload (Solidity-ready) | — |
| 4 | Custom contract schemas | — |
| 5 | scikit-learn `IEQFeatureExtractor` | `[ml]` |
| 6 | PyTorch `IEQTimeSeriesDataset` | `[torch]` |
| 7 | TensorFlow/Keras `IEQPreprocessingLayer` | `[keras]` |

In [1]:
import numpy as np
import pandas as pd
import json
import traceback

from comfio import (
    SensorData,
    evaluate_thermal,
    evaluate_visual,
    evaluate_acoustic,
    evaluate_iaq,
    calculate_global_ieq,
    calculate_compliance,
)

# Simulate 24 hours of sensor data
rng = np.random.default_rng(42)
n = 24
df = pd.DataFrame({
    "temperature": 23 + rng.normal(0, 1.5, n),
    "radiant_temp": 23 + rng.normal(0, 1.5, n),
    "rh": 50 + rng.normal(0, 5, n),
    "air_speed": 0.1 + rng.normal(0, 0.02, n),
    "lux": 450 + rng.normal(0, 80, n),
    "laeq": 45 + rng.normal(0, 6, n),
    "co2": 700 + rng.normal(0, 120, n),
})

# Evaluate all domains
thermal = evaluate_thermal(
    tdb=df["temperature"].values, tr=df["radiant_temp"].values,
    vr=df["air_speed"].values, rh=df["rh"].values,
    met=1.2, clo=0.5, category="B",
)
visual = evaluate_visual(illuminance=df["lux"].values, task_type="office_writing")
acoustic = evaluate_acoustic(laeq=df["laeq"].values, nc_level="NC-35")
iaq = evaluate_iaq(co2=df["co2"].values, threshold_level="good")

ieq_result = calculate_global_ieq(
    thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
)

print(f"24-hour IEQ Index: {np.round(ieq_result.index, 1)}")
print(f"Mean IEQ: {np.mean(ieq_result.index):.1f}")

24-hour IEQ Index: [78.5 68.2 80.6 71.5 61.7 71.9 74.7 83.6 78.1 76.2 78.  81.  73.8 69.2
 84.7 83.3 77.4 67.7 66.1 72.  72.5 81.  88.9 72.5]
Mean IEQ: 75.5


## Part 1: Smart Contract Integration

### 1. Compliance Report

`calculate_compliance()` converts the IEQ Index array into time-based compliance metrics. The `ComplianceReport` dataclass holds all the fields needed for performance contract verification.

In [2]:
# Generate compliance report at threshold = 70
report = calculate_compliance(ieq_result, threshold=70.0)

print("Compliance Report (threshold = 70):")
print(f"  IEQ avg:           {report.ieq_index_avg:.1f}")
print(f"  IEQ min/max:       {report.ieq_index_min:.1f} / {report.ieq_index_max:.1f}")
print(f"  IEQ std:           {report.ieq_index_std:.1f}")
print(f"  Compliance rate:   {report.compliance_rate_pct:.1f}%")
print(f"  Compliant hours:   {report.compliant_hours:.0f} / {report.total_hours:.0f}")
print(f"  Period:            {report.period_start:.0f} → {report.period_end:.0f}")
print(f"  Domains:           {report.domains}")
print(f"  Weights:           {report.weights_used}")

print(f"\nPer-domain compliance (% of hours with score >= 80):")
for domain, rate in report.domain_compliance.items():
    print(f"  {domain:12s}: {rate:.1f}%")

print(f"\nPer-domain average scores:")
for domain, avg in report.domain_scores_avg.items():
    print(f"  {domain:12s}: {avg:.1f}")

Compliance Report (threshold = 70):
  IEQ avg:           75.5
  IEQ min/max:       61.7 / 88.9
  IEQ std:           6.4
  Compliance rate:   79.2%
  Compliant hours:   19 / 24
  Period:            1783027350 → 1783113750
  Domains:           ['thermal', 'visual', 'acoustic', 'iaq']
  Weights:           {'thermal': 0.4, 'visual': 0.2, 'acoustic': 0.15, 'iaq': 0.25}

Per-domain compliance (% of hours with score >= 80):
  thermal     : 83.3%
  visual      : 70.8%
  acoustic    : 4.2%
  iaq         : 50.0%

Per-domain average scores:
  thermal     : 85.6
  visual      : 85.1
  acoustic    : 34.1
  iaq         : 76.8


In [3]:
# Export as full JSON (all fields, for storage / API / oracle)
json_str = report.to_json()
print(f"Full JSON report ({len(json_str)} chars):\n")
print(json.dumps(json.loads(json_str), indent=2))

Full JSON report (829 chars):

{
  "period_start": 1783027350.4044147,
  "period_end": 1783113750.4044147,
  "ieq_index_avg": 75.54737349025675,
  "ieq_index_min": 61.72455103738686,
  "ieq_index_max": 88.90534646368798,
  "ieq_index_std": 6.446390317367879,
  "compliance_rate_pct": 79.16666666666666,
  "threshold": 70.0,
  "total_hours": 24.0,
  "compliant_hours": 19.0,
  "domain_compliance": {
    "thermal": 83.33333333333334,
    "visual": 70.83333333333334,
    "acoustic": 4.166666666666666,
    "iaq": 50.0
  },
  "domain_scores_avg": {
    "thermal": 85.55833333333334,
    "visual": 85.05232413284497,
    "acoustic": 34.14388695451654,
    "iaq": 76.76796914870776
  },
  "weights_used": {
    "thermal": 0.4,
    "visual": 0.2,
    "acoustic": 0.15,
    "iaq": 0.25
  },
  "domains": [
    "thermal",
    "visual",
    "acoustic",
    "iaq"
  ]
}


### 2. Contract Schema & ABI

`ContractSchema` defines the mapping between comfio compliance data and Solidity function parameters. The default schema targets an `IEQComplianceOracle` contract with a `submitCompliance` function.

In [4]:
from comfio.performance.contract_schema import default_compliance_schema

schema = default_compliance_schema()

print(f"Contract:  {schema.contract_name}")
print(f"Function:  {schema.function_name}")
print(f"\nFields ({len(schema.fields)}):")
print(f"  {'Name':<25s} {'Solidity Type':<12s} {'Source'}")
print(f"  {'-'*70}")
for flapjack in schema.fields:
    print(f"  {flapjack.name:<25s} {flapjack.solidity_type:<12s} {flapjack.source}")

# Generate ABI fragment (ready for web3.py / ethers.js)
abi = schema.to_abi()
print(f"\nABI fragment:")
print(json.dumps(abi, indent=2))

Contract:  IEQComplianceOracle
Function:  submitCompliance

Fields (10):
  Name                      Solidity Type Source
  ----------------------------------------------------------------------
  periodStart               uint256      report.period_start
  periodEnd                 uint256      report.period_end
  ieqIndexAvg               uint8        report.ieq_index_avg
  complianceRatePct         uint8        report.compliance_rate_pct
  thermalCompliant          bool         report.domain_compliance.thermal
  visualCompliant           bool         report.domain_compliance.visual
  acousticCompliant         bool         report.domain_compliance.acoustic
  iaqCompliant              bool         report.domain_compliance.iaq
  totalOccupiedHours        uint32       report.total_occupied_hours
  compliantHours            uint32       report.compliant_hours

ABI fragment:
{
  "name": "submitCompliance",
  "type": "function",
  "stateMutability": "nonpayable",
  "inputs": [
    {
      

### 3. Contract Payload (Solidity-ready)

`to_contract_payload()` maps the `ComplianceReport` to Solidity-compatible types (uint256, uint8, bool, uint32). `to_contract_json()` produces the JSON string version.

In [5]:
# Solidity-ready payload (types converted: float -> int/bool)
payload = report.to_contract_payload()
print("Contract payload (Solidity-compatible):")
print(f"  {'Field':<25s} {'Value':<20s} {'Type'}")
print(f"  {'-'*55}")
for key, value in payload.items():
    print(f"  {key:<25s} {str(value):<20s} {type(value).__name__}")

# JSON string version (ready to send to oracle / web3.py)
contract_json = report.to_contract_json()
print(f"\nContract JSON ({len(contract_json)} chars):")
print(contract_json)

Contract payload (Solidity-compatible):
  Field                     Value                Type
  -------------------------------------------------------
  periodStart               1783027350           int
  periodEnd                 1783113750           int
  ieqIndexAvg               76                   int
  complianceRatePct         79                   int
  thermalCompliant          True                 bool
  visualCompliant           False                bool
  acousticCompliant         False                bool
  iaqCompliant              False                bool
  totalOccupiedHours        24                   int
  compliantHours            19                   int

Contract JSON (269 chars):
{
  "periodStart": 1783027350,
  "periodEnd": 1783113750,
  "ieqIndexAvg": 76,
  "complianceRatePct": 79,
  "thermalCompliant": true,
  "visualCompliant": false,
  "acousticCompliant": false,
  "iaqCompliant": false,
  "totalOccupiedHours": 24,
  "compliantHours": 19
}


### 4. Custom Contract Schema

You can define your own schema with different field mappings, contract names, or Solidity types. This lets you target any smart contract that accepts IEQ compliance data.

In [6]:
from comfio.performance.contract_schema import ContractSchema, ContractField

# Custom schema for a "GreenBuildingReward" contract
custom_schema = ContractSchema(
    contract_name="GreenBuildingReward",
    function_name="claimReward",
    fields=[
        ContractField(
            name="periodStart",
            solidity_type="uint256",
            description="Reporting period start timestamp.",
            source="report.period_start",
        ),
        ContractField(
            name="periodEnd",
            solidity_type="uint256",
            description="Reporting period end timestamp.",
            source="report.period_end",
        ),
        ContractField(
            name="ieqScore",
            solidity_type="uint8",
            description="Average IEQ Index (0-100).",
            source="report.ieq_index_avg",
        ),
        ContractField(
            name="isCompliant",
            solidity_type="bool",
            description="Whether IEQ >= threshold for the period.",
            source="report.compliance_rate_pct",
        ),
        ContractField(
            name="hours",
            solidity_type="uint32",
            description="Total occupied hours.",
            source="report.total_occupied_hours",
        ),
    ],
)

print(f"Custom contract: {custom_schema.contract_name}")
print(f"Function:        {custom_schema.function_name}")
print(f"\nABI:")
print(json.dumps(custom_schema.to_abi(), indent=2))

# Generate payload using the custom schema
custom_payload = report.to_contract_payload(schema=custom_schema)
print(f"\nCustom payload:")
for key, value in custom_payload.items():
    print(f"  {key:<15s}: {value} ({type(value).__name__})")

Custom contract: GreenBuildingReward
Function:        claimReward

ABI:
{
  "name": "claimReward",
  "type": "function",
  "stateMutability": "nonpayable",
  "inputs": [
    {
      "name": "periodStart",
      "type": "uint256",
      "internalType": "uint256"
    },
    {
      "name": "periodEnd",
      "type": "uint256",
      "internalType": "uint256"
    },
    {
      "name": "ieqScore",
      "type": "uint8",
      "internalType": "uint8"
    },
    {
      "name": "isCompliant",
      "type": "bool",
      "internalType": "bool"
    },
    {
      "name": "hours",
      "type": "uint32",
      "internalType": "uint32"
    }
  ],
  "outputs": []
}

Custom payload:
  periodStart    : 1783027350 (int)
  periodEnd      : 1783113750 (int)
  ieqScore       : 76 (int)
  isCompliant    : 79 (int)
  hours          : 24 (int)


### 5. Multiple Thresholds & Period Comparison

A real performance contract might compare compliance across different thresholds or time periods.

In [7]:
# Compare compliance at different thresholds
print(f"{'Threshold':>10s}  {'Rate':>8s}  {'Hours':>10s}  {'Payload ieqIndexAvg'}")
print(f"  {'-'*55}")

for threshold in [50, 60, 70, 80, 90]:
    r = calculate_compliance(ieq_result, threshold=float(threshold))
    p = r.to_contract_payload()
    print(f"  {threshold:>8d}   {r.compliance_rate_pct:>6.1f}%  {r.compliant_hours:>4.0f}/{r.total_hours:.0f}h   ieqIndexAvg={p['ieqIndexAvg']}")

# Custom period timestamps
import time
now = time.time()
report_period = calculate_compliance(
    ieq_result, threshold=75.0,
    period_start=now - 86400,  # 24h ago
    period_end=now,
)
print(f"\nCustom period report:")
print(f"  Start: {report_period.period_start:.0f} ({time.strftime('%Y-%m-%d %H:%M', time.localtime(report_period.period_start))})")
print(f"  End:   {report_period.period_end:.0f} ({time.strftime('%Y-%m-%d %H:%M', time.localtime(report_period.period_end))})")
print(f"  Rate:  {report_period.compliance_rate_pct:.1f}%")

 Threshold      Rate       Hours  Payload ieqIndexAvg
  -------------------------------------------------------
        50    100.0%    24/24h   ieqIndexAvg=76
        60    100.0%    24/24h   ieqIndexAvg=76
        70     79.2%    19/24h   ieqIndexAvg=76
        80     29.2%     7/24h   ieqIndexAvg=76
        90      0.0%     0/24h   ieqIndexAvg=76

Custom period report:
  Start: 1783027350 (2026-07-02 23:22)
  End:   1783113750 (2026-07-03 23:22)
  Rate:  50.0%


## Part 2: ML / DL Integration

comfio provides adapters for three ML frameworks. Each wraps the IEQ calculation pipeline into the framework's native API so you can use IEQ features directly in model training.

| Framework | Class | Extra |
|-----------|-------|-------|
| scikit-learn | `IEQFeatureExtractor` | `pip install comfio[ml]` |
| PyTorch | `IEQTimeSeriesDataset` | `pip install comfio[torch]` |
| TensorFlow/Keras | `IEQPreprocessingLayer` | `pip install comfio[keras]` |

**Graceful degradation**: If a framework is not installed, the cell prints a helpful message instead of crashing.

In [8]:
def try_run(label, fn):
    """Run a function, catching ImportError for missing ML extras."""
    try:
        result = fn()
        print(f"  [OK] {label}")
        return result
    except ImportError as e:
        print(f"  [SKIP] {label}")
        print(f"        {e}")
        return None
    except Exception:
        print(f"  [ERROR] {label}")
        traceback.print_exc()
        return None

# Larger dataset for ML demos (100 timestamps)
rng = np.random.default_rng(99)
n_ml = 100
df_ml = pd.DataFrame({
    "temperature": 23 + rng.normal(0, 1.5, n_ml),
    "radiant_temp": 23 + rng.normal(0, 1.5, n_ml),
    "rh": 50 + rng.normal(0, 5, n_ml),
    "air_speed": 0.1 + rng.normal(0, 0.02, n_ml),
    "lux": 450 + rng.normal(0, 80, n_ml),
    "laeq": 45 + rng.normal(0, 6, n_ml),
    "co2": 700 + rng.normal(0, 120, n_ml),
    "metabolic_rate_met": np.full(n_ml, 1.2),
    "clothing_insulation_clo": np.full(n_ml, 0.5),
})
print(f"ML dataset: {df_ml.shape[0]} rows, {df_ml.shape[1]} columns")

ML dataset: 100 rows, 9 columns


### 5. scikit-learn: `IEQFeatureExtractor`

Implements `fit`/`transform` interface for use in `sklearn.pipeline.Pipeline`. Extracts IEQ Index + per-domain scores as feature columns.

In [9]:
def demo_sklearn():
    from comfio.ml.sklearn_transformers import IEQFeatureExtractor

    # Extract IEQ features (ieq_index + 4 domain scores = 5 columns)
    extractor = IEQFeatureExtractor(
        include_ieq_index=True,
        include_domain_scores=True,
        thermal_category="B",
        visual_task_type="office_writing",
        acoustic_nc_level="NC-35",
        iaq_threshold_level="good",
    )
    extractor.fit(df_ml)
    features = extractor.transform(df_ml)

    print(f"IEQFeatureExtractor:")
    print(f"  Input:  {df_ml.shape}")
    print(f"  Output: {features.shape}")
    print(f"  Features: {list(extractor.get_feature_names_out())}")
    print(f"  Range:   [{features.min():.1f}, {features.max():.1f}]")
    print(f"  First row: {np.round(features[0], 1)}")

    # Use in a sklearn Pipeline
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split

    # Target: predict IEQ index from raw sensors (demo)
    X_train, X_test = train_test_split(df_ml, test_size=0.2, random_state=42)
    y_train = extractor.fit(X_train).transform(X_train)[:, 0]  # ieq_index column

    pipe = Pipeline([
        ("ieq", IEQFeatureExtractor(include_domain_scores=False)),
        ("model", RandomForestRegressor(n_estimators=10, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    print(f"\n  Pipeline: IEQFeatureExtractor → RandomForestRegressor")
    print(f"  Train: {len(X_train)}, Test: {len(X_test)}")
    print(f"  Predictions (first 5): {np.round(predictions[:5], 1)}")
    return features

sklearn_result = try_run("scikit-learn IEQFeatureExtractor", demo_sklearn)

IEQFeatureExtractor:
  Input:  (100, 9)
  Output: (100, 4)
  Features: [np.str_('ieq_index'), np.str_('visual_score'), np.str_('acoustic_score'), np.str_('iaq_score')]
  Range:   [0.0, 100.0]
  First row: [68.  99.7  0.  83.4]



  Pipeline: IEQFeatureExtractor → RandomForestRegressor
  Train: 80, Test: 20
  Predictions (first 5): [66.9 85.1 67.  63.9 78.1]
  [OK] scikit-learn IEQFeatureExtractor


### 6. PyTorch: `IEQTimeSeriesDataset`

Wraps sensor data into windowed samples for sequence models (LSTM, Transformer). Compatible with `torch.utils.data.DataLoader`.

In [10]:
def demo_torch():
    from comfio.ml.torch_dataset import IEQTimeSeriesDataset
    from torch.utils.data import DataLoader

    # Windowed dataset: 24-timestep windows, stride=1
    dataset = IEQTimeSeriesDataset(
        df_ml, window_size=24, stride=1,
        include_raw=True, include_ieq=True,
    )
    print(f"IEQTimeSeriesDataset:")
    print(f"  Total rows:     {len(df_ml)}")
    print(f"  Window size:    {dataset.window_size}")
    print(f"  N windows:      {len(dataset)}")
    print(f"  Raw features:   {dataset.raw_feature_names}")

    sample = dataset[0]
    print(f"\n  Sample[0] keys: {list(sample.keys())}")
    print(f"  raw shape:      {sample['raw'].shape}")
    print(f"  ieq_index shape: {sample['ieq_index'].shape}")
    print(f"  ieq_index range: [{sample['ieq_index'].min():.1f}, {sample['ieq_index'].max():.1f}]")
    print(f"  domain_scores:   {list(sample['domain_scores'].keys())}")

    # DataLoader for batch training
    loader = DataLoader(dataset, batch_size=8, shuffle=False)
    batch = next(iter(loader))
    print(f"\n  DataLoader batch:")
    print(f"    ieq_index: {batch['ieq_index'].shape}")
    print(f"    raw:       {batch['raw'].shape}")
    return dataset

torch_result = try_run("PyTorch IEQTimeSeriesDataset", demo_torch)

IEQTimeSeriesDataset:
  Total rows:     100
  Window size:    24
  N windows:      77
  Raw features:   ['air_temp_c', 'relative_humidity_pct', 'air_velocity_ms', 'illuminance_lux', 'co2_ppm', 'noise_laeq_db', 'metabolic_rate_met', 'clothing_insulation_clo']

  Sample[0] keys: ['raw', 'ieq_index', 'domain_scores']
  raw shape:      (24, 8)
  ieq_index shape: (24,)
  ieq_index range: [59.2, 83.9]
  domain_scores:   ['visual', 'acoustic', 'iaq']

  DataLoader batch:
    ieq_index: torch.Size([8, 24])
    raw:       torch.Size([8, 24, 8])
  [OK] PyTorch IEQTimeSeriesDataset


### 7. TensorFlow/Keras: `IEQPreprocessingLayer`

Wraps IEQ feature extraction into a Keras layer interface. Accepts DataFrames (eager) or tensors (deferred via `py_function`).

In [11]:
def demo_keras():
    from comfio.ml.keras_adapter import IEQPreprocessingLayer

    # Build preprocessing layer
    layer = IEQPreprocessingLayer(
        include_ieq_index=True,
        include_domain_scores=True,
        thermal_category="B",
        visual_task_type="office_writing",
    )
    layer.adapt(df_ml)

    # Eager computation from DataFrame
    features = layer(df_ml)
    print(f"IEQPreprocessingLayer:")
    print(f"  Input:    DataFrame {df_ml.shape}")
    print(f"  Output:   {features.shape} (tf.Tensor)")
    print(f"  Features: {layer._feature_names}")
    print(f"  Range:    [{float(features.numpy().min()):.1f}, {float(features.numpy().max()):.1f}]")
    print(f"  First row: {features.numpy()[0]}")
    return features

keras_result = try_run("TensorFlow/Keras IEQPreprocessingLayer", demo_keras)

  [SKIP] TensorFlow/Keras IEQPreprocessingLayer
        No module named 'tensorflow'


## Summary

### Smart Contract Integration (no extras needed)
- `calculate_compliance()` → `ComplianceReport` with IEQ stats, compliance rates, domain breakdowns
- `to_json()` → full JSON for storage/APIs
- `to_contract_payload()` → Solidity-compatible types (uint256, uint8, bool, uint32)
- `to_contract_json()` → JSON string ready for oracle submission
- `ContractSchema` → define custom field mappings for any Solidity function
- `schema.to_abi()` → ABI fragment for web3.py / ethers.js

### ML/DL Integration (optional extras)
```bash
pip install comfio[ml]      # scikit-learn IEQFeatureExtractor
pip install comfio[torch]   # PyTorch IEQTimeSeriesDataset
pip install comfio[keras]   # TensorFlow IEQPreprocessingLayer
```

Or all at once: `pip install comfio[all]`

Full documentation: **GUIDE.md Sections 8–12**

# comfio Smart Contract & ML Integration Tutorial

**Part 1** requires no optional extras — just the core package.
**Part 2** requires `pip install comfio[ml]` (scikit-learn), `comfio[torch]` (PyTorch), or `comfio[keras]` (TensorFlow).

This notebook covers:

| Section | Topic | Extra Required |
|---------|-------|---------------|
| 1 | Compliance reporting & JSON export | — |
| 2 | Smart contract schema & ABI generation | — |
| 3 | Contract payload (Solidity-ready) | — |
| 4 | Custom contract schemas | — |
| 5 | scikit-learn `IEQFeatureExtractor` | `[ml]` |
| 6 | PyTorch `IEQTimeSeriesDataset` | `[torch]` |
| 7 | TensorFlow/Keras `IEQPreprocessingLayer` | `[keras]` |